# Optimizing Agentic AI Collaboration v5.2.0 — BoN-MAV
## Verifier: Ministral-3-3B-Instruct | Candidates: E6 | Aspects: MATH-S1 | MATH-500

**Model:** `mistralai/Ministral-3-3B-Instruct-2512-BF16` (verifier only) with 4-bit NF4 quantization via bitsandbytes.
**Hardware target:** Nvidia T4 GPU (Kaggle / Colab free tier).

## What this notebook is

This notebook runs one **BoN-MAV** configuration: `ministral-3-3b` verifies the 3-candidate `E6` ensemble (`ministral-3-3b` × 3) on **MATH-500**, using the **`MATH-S1`** aspect subset (4 AVs: 1 `direct` + 2 `reasoning` + 1 `step_by_step`). This is the **self-improvement** — `ministral-3-3b` verifies its own 3-way self-generated ensemble.

`E6` never appears in `MAJ_ENSEMBLES` — this is a self-improvement configuration, not a MAJ pairing — but `sample_slots(E6, ..., rep=0)` is still a well-defined, independently reproducible draw: `sample_slots` only needs an `Ensemble`, not a prior MAJ run having executed it. This (model, dataset) pair draws **exactly one** N=3 sample — never resampled or repeated. Only the verifier is ever live: **12 verifier calls per problem** (3 candidates × 4 aspect verifiers), each an independent `chain.invoke` (no shared KV-cache across aspects).

**⚠️ GPU budget — read before running.** This configuration's worst-case estimate is **~6.6 h (optimistic, 30 tok/s) to ~12.5 h (conservative, 15 tok/s)** for all 100 problems — likely exceeding a single Kaggle session. This notebook is **resumable by design**: it re-logs its growing verdicts+selections artifact every ~10 problems and can restart from `RESUME_FROM_REF` (§3) rather than `problem_id=0`. **Before scheduling a real multi-hour session**, extend the `v3.0.0` smoke test's Stage C calibration to ~10–20 real problems on this exact verifier/dataset and use the *measured* mean tokens-per-strategy to refine this estimate — the worst case assumes every call runs to its `max_new_tokens` cap, which none of the aspects' own instructions actually require.

## Relation with the project WPs

**WP2 - T4: L1 multi-agent protocol — BoN-MAV.** Consumes `ministral-3-3b`'s own SC generation artifact 3 times over (the `E6` ensemble is `ministral-3-3b` × 3) — the model verifying its own independently-sampled outputs. Latency/token/VRAM (via `utils.voting.compose_generation_metrics`, split `reused_gen_*`/`live_verify_*`) feed the WP3 - T1 compute-accuracy trade-off analysis.

---
### Before running (Kaggle)

0. Upload this notebook to a Kaggle kernel (or open it directly if already there).
1. In **Settings**, enable **Accelerator: GPU T4 x2** (the notebook pins itself to one GPU).
2. Add secrets via **Add-ons → Secrets**: `GITHUB_TOKEN`, `HF_TOKEN`, `WANDB_API_KEY`.
3. **Before clicking Run All**, fill in §3's `PRODUCER_REFS` (pinned SC artifact versions for `ministral-3-3b` on `math500`) and, if resuming a killed session, `RESUME_FROM_REF`.
4. Click **Run All**. If the session is killed, note the last-printed re-log ref, set `RESUME_FROM_REF` to it, and re-run — the loop skips every already-logged `problem_id`.

### Before running (Colab)

0. **Copy this notebook** → File → Save a copy in Drive
1. **Enable GPU** → Runtime → Change runtime type → T4 GPU → Save
2. **Add Colab Secrets**: `HF_TOKEN`, `WANDB_API_KEY`, `GITHUB_TOKEN` (Notebook access enabled)
3. Fill in `PRODUCER_REFS`/`RESUME_FROM_REF` in §3 before running, same as above.

## 1. Setup and Installation

We install the libraries required by the repo (Hugging Face stack, `langchain*`, `langgraph`, `datasets`, `wandb`, `math-verify`). If running on Colab/Kaggle, the repo (with the `utils` helper modules) is cloned first.

In [ ]:
# ── Environment detection: clone private repo ────────────────
import sys, os

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    ENVIRONMENT = "kaggle"
elif "COLAB_RELEASE_TAG" in os.environ or "COLAB_BACKEND_VERSION" in os.environ:
    ENVIRONMENT = "colab"
else:
    ENVIRONMENT = "local"


if ENVIRONMENT == "colab":
    repo_root = "/content/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from google.colab import userdata  # type: ignore
                os.environ.setdefault("GITHUB_TOKEN", userdata.get("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()

            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

elif ENVIRONMENT == "kaggle":

    # Kaggle provides 2x T4 GPUs; restrict to a single GPU so that
    # experiments run under the same resource constraints as Colab.
    # This MUST happen before any torch / CUDA import.
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    repo_root = "/kaggle/working/tfg-agentic-ai"
    if not os.path.isdir(repo_root):
        try:
            try:
                from kaggle_secrets import UserSecretsClient  # type: ignore
                _secrets = UserSecretsClient()
                os.environ.setdefault("GITHUB_TOKEN", _secrets.get_secret("GITHUB_TOKEN") or "")
            except ImportError:
                from dotenv import load_dotenv  # type: ignore
                load_dotenv()
            from kaggle_secrets import UserSecretsClient  # type: ignore
            repo_url = f"https://{os.environ.get('GITHUB_TOKEN')}@github.com/andreufc/tfg-agentic-ai.git"
            import subprocess  # type: ignore
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_root], check=True)
            print(f"✅ Repo cloned to {repo_root}")
        except Exception as e:
            print(f"Error cloning repo: {e}")
    else:
        print(f"ℹ️  Repo already present at {repo_root}")
    os.chdir(os.path.join(repo_root, "notebooks"))

else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"repo_root = {repo_root}")

In [ ]:
# Only necessary if running in Google Colab, but won't cause issues if run locally
%cd {repo_root}
%pip install -r requirements.txt

## 2. Authentication via Colab / Kaggle Secrets

We pull `HF_TOKEN` and `WANDB_API_KEY` from Colab userdata or Kaggle Secrets — no tokens in code.

In [ ]:
from utils.config import load_env

print(f"Runtime environment: {ENVIRONMENT}")

load_env(environment=ENVIRONMENT)

## 3. Config

`ENSEMBLE` (`E6`) and `SUBSET_ID` (`MATH-S1`) fix this notebook's configuration; `PRODUCER_REFS` pins the 3 candidate models' SC artifacts **explicitly** (`entity/project/name:vN`, never `:latest`). `RESUME_FROM_REF` — leave `None` for a fresh run; set to this notebook's own last-printed verdicts-artifact ref to resume a killed session.

In [ ]:
import random

from utils.ensembles import E6, SLOT_SAMPLE_SEED, bonmav_matrices_artifact_name, bonmav_run_name, sample_slots, slot_agent_ids
from utils.registry import MODELS, generations_artifact_name, get_model, problem_ref
from utils.mav import MAV_ASPECT_SUBSETS

VERIFIER_KEY = "ministral-3-3b"
ENSEMBLE = E6
SUBSET_ID = "MATH-S1"
DATASET = "math500"
ASPECT_IDS = MAV_ASPECT_SUBSETS[DATASET][SUBSET_ID]

PROJECT = "agentic-final-results" if (ENVIRONMENT in ("kaggle", "collab")) else "dev-agentic-baselines"
print(f"W&B Project: {PROJECT}")

WANDB_ENTITY = "tfg-optimizing-agentic-ai"
SC_PROJECT = "agentic-final-results"

CANDIDATE_MODEL_KEYS = list(dict.fromkeys(ENSEMBLE.slots))
PRODUCER_REFS = {}
for model_key in CANDIDATE_MODEL_KEYS:
    version = "v9" if "ministral" in model_key else "v4"
    PRODUCER_REFS[model_key] = f"{WANDB_ENTITY}/{SC_PROJECT}/{generations_artifact_name(model_key, DATASET)}:{version}"

DATA_SEED = 789
EVAL_SIZE = 100
GEN_COMMON = {"temperature": 0.7, "top_p": 0.95, "top_k": 40}  # matches the L1 GEN params
TIE_BREAK_SEED = 2026
LOG_EVERY_N_PROBLEMS = 10  # re-log the resumable verdicts artifact this often

# Set to this notebook's own last-printed verdicts-artifact ref to resume a killed
# session instead of restarting at problem_id=0; leave None for a fresh run.
RESUME_FROM_REF = None

VERDICTS_ARTIFACT_NAME = f"l1-bonmav-{VERIFIER_KEY}-{ENSEMBLE.ensemble_id}-{SUBSET_ID}-{DATASET}-verifications"

print(f"Verifier: {VERIFIER_KEY}  |  Candidates: {ENSEMBLE.ensemble_id} {list(ENSEMBLE.slots)}  |  "
      f"Subset: {SUBSET_ID}  |  Dataset: {DATASET}")
print(f"Aspect ids ({len(ASPECT_IDS)}): {ASPECT_IDS}")
print(f"\nPinned candidate producer refs:")
for mk, ref in PRODUCER_REFS.items():
    print(f"  {mk}: {ref}")

## 4. Data Loading

The evaluation subset is drawn with `random.seed(789)`, identical to every SC/MAJ notebook's own subset — so `problem_id` identity and `sample_slots`' draws all agree across protocols.

In [ ]:
from utils.data import load_math500
from utils.voting import MATH500_SPEC

SPEC = MATH500_SPEC
math500_dataset = load_math500(split="test")

random.seed(DATA_SEED)
test_sample = random.sample(list(math500_dataset), EVAL_SIZE)

print(f"✅ MATH-500 dataset loaded with {len(math500_dataset)} examples.")
print(f"✅ {len(test_sample)} problems sampled (seed={DATA_SEED})")

## 5. Reuse — Candidate Model Keys

`CANDIDATE_MODEL_KEYS` (from §3) is every distinct model in `ENSEMBLE`. The actual producer-artifact fetch (`utils.persistence.fetch_and_validate_stores`: `load_generations_artifact` → `build_generation_store` → `validate_alignment`, the same reuse pattern `v4.1.0` and the `v3.0.0` smoke test's round-trip stage already use) now happens in §8, immediately after the real BoN-MAV run is opened, via `run=run` — so the run that actually consumes the data also records its own producer→consumer lineage, instead of an earlier anonymous download. No model is loaded here — this is pure download + validation, deferred to §8.

## 6. Load Verifier Model

Only the verifier is ever resident in VRAM — every candidate is replayed, never regenerated — so `batch_size=1` and the single-model VRAM budget hold throughout, identical in shape to the SC notebooks' own model-loading cell.

In [ ]:
from utils.models import load_model_and_tokenizer
from utils.warmup import warmup_model

verifier_cfg = get_model(VERIFIER_KEY)
model, tokenizer = load_model_and_tokenizer(verifier_cfg["model_id"], quantize_4bit=True)
print(f"✅ Verifier {VERIFIER_KEY} loaded on {model.device} — dtype {model.dtype}")

warmup_model(
    model, tokenizer,
    max_new_tokens=256, n_iterations=2,
    temperature=GEN_COMMON["temperature"], top_p=GEN_COMMON["top_p"], top_k=GEN_COMMON["top_k"],
)

## 7. Verify Function and Select Function

`verify_fn` is a `timed_invoke`-based closure over the loaded verifier: it renders the transcribed aspect prompt (`utils.mav.MAV_ASPECTS`) with `{task}`/`{candidate}`, dispatching on the aspect's `strategy` for `max_new_tokens` (128 `direct` / 512 `reasoning` / 1024 `step_by_step`). `select_fn` wraps `utils.mav.select_bon_candidate` with this benchmark's `SPEC` bound (`build_bon_mav_graph` calls it as `select_fn(n_candidates, approvals, candidate_answers, rng=rng)`, with no `spec` parameter of its own).

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from utils.agents import build_hf_langchain_llm
from utils.instrumentation import timed_invoke, tokens_per_second
from utils.mav import MAV_ASPECTS, parse_verdict, select_bon_candidate


def make_verify_fn(model, tokenizer, model_key, dataset_key, sampling):
    """A timed_invoke-based closure over the (single, already-loaded) verifier
    model, dispatching on the aspect's strategy for max_new_tokens — one llm
    built per distinct max_new_tokens value, reused across problems/aspects
    that share it."""
    llm_by_max_tokens = {}

    def get_llm(max_new_tokens):
        if max_new_tokens not in llm_by_max_tokens:
            llm_by_max_tokens[max_new_tokens] = build_hf_langchain_llm(
                model=model, tokenizer=tokenizer, max_new_tokens=max_new_tokens,
                temperature=sampling["temperature"], top_p=sampling["top_p"], top_k=sampling["top_k"],
                return_full_text=False,
            )
        return llm_by_max_tokens[max_new_tokens]

    def verify(task, candidate_answer, aspect_id):
        aspect = MAV_ASPECTS[dataset_key][aspect_id]
        prompt_template = PromptTemplate.from_template(aspect["prompt"])
        llm = get_llm(aspect["max_new_tokens"])
        chain = prompt_template | llm | StrOutputParser()
        r = timed_invoke(chain, prompt_template, tokenizer, {"task": task, "candidate": candidate_answer})
        return {
            "candidate_idx": -1, "aspect_id": aspect_id, "verdict": parse_verdict(r["output_text"]),
            "raw_output": r["output_text"], "latency_s": r["latency_s"],
            "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
            "total_tokens": r["total_tokens"], "gpu_peak_mb": r["gpu_peak_mb"],
            "model_key": model_key,
        }
    return verify


def make_select_fn(spec):
    def select_fn(n_candidates, approvals, candidate_answers, *, rng):
        return select_bon_candidate(n_candidates, approvals, candidate_answers, spec=spec, rng=rng)
    return select_fn


verify_fn = make_verify_fn(model, tokenizer, VERIFIER_KEY, DATASET, GEN_COMMON)
select_fn = make_select_fn(SPEC)
print("✅ verify_fn and select_fn ready.")

## 8. Resumable BoN-MAV Run Loop

`run = init_wandb(...)` opens the real run first, then `fetch_and_validate_stores(CANDIDATE_MODEL_KEYS, ..., run=run)` downloads + validates every candidate producer artifact — real producer→consumer lineage recorded on *this* run (`run.config` is stamped with `alignment_validated`/`alignment_record_counts` afterward via `run.config.update(...)`, since those aren't known before the fetch). `RESUME_FROM_REF`, if set, is then loaded the same way (`load_generations_artifact(RESUME_FROM_REF, run=run)`). Per problem: `sample_slots(ENSEMBLE, ..., rep=0)` (a well-defined, independently reproducible draw — `E6`/`E7` never appear in `MAJ_ENSEMBLES`, so this is not a MAJ pairing, but `sample_slots` only needs an `Ensemble`) → 3 candidate `AgentSpec`s (`mode="replay"`) → `build_bon_mav_graph(...)` → `graph.invoke(...)` → score `out["selection"]`'s pick against gold **outside** the graph → `compose_generation_metrics(out["samples"] + out["approvals"], tokens_per_s_rule="mean")`, plus the `reused_gen_*`/`live_verify_*` split computed separately on each half.

**Resumable, not just checkpointed**: every `LOG_EVERY_N_PROBLEMS` problems (and once more at the end), the full record list so far is re-logged under one growing artifact name (`log_generations_artifact` — the identical crash-resumption mechanism the SC generation pass uses per-seed). If this session is killed, the last-printed ref is the one to set as `RESUME_FROM_REF` (§3) — the loop then skips every already-logged `problem_id`.

In [ ]:
import time
import wandb
from utils.logging_utils import init_wandb, define_wandb_chart_only_metrics, log_metrics, log_stat_summary, finish_wandb
from utils.graphs import AgentSpec, build_bon_mav_graph
from utils.voting import compose_generation_metrics
from utils.persistence import artifact_ref, load_generations_artifact, log_generations_artifact, fetch_and_validate_stores

# Protocol tag format: {protocol name}@N/M<N> = BoN-MAV with N=3 generators
# (candidates) and M=4 Aspect Verifiers per subset (see utils/mav.py). The
# logged tag string itself is unchanged.
run = init_wandb(
    run_name=bonmav_run_name(VERIFIER_KEY, ENSEMBLE.ensemble_id, SUBSET_ID, DATASET),
    project=PROJECT,
    config={
        "protocol": "BoN-MAV@3/M4", "verifier_key": VERIFIER_KEY, "candidate_ensemble": ENSEMBLE.ensemble_id,
        "aspect_subset": SUBSET_ID, "aspect_ids": ASPECT_IDS, "eval_dataset": DATASET, "eval_size": EVAL_SIZE,
        "data_seed": DATA_SEED, "tie_break_seed": TIE_BREAK_SEED, "slot_sample_seed": SLOT_SAMPLE_SEED,
        "producer_artifacts": PRODUCER_REFS, "resumed_from": RESUME_FROM_REF,
    },
    tags=["l1", "bon-mav", VERIFIER_KEY, ENSEMBLE.ensemble_id, SUBSET_ID, DATASET],
)
# Own W&B run fetches its own copy of every candidate producer artifact via
# load_generations_artifact(run=run) — real producer->consumer lineage
# recorded on THIS run, not a bare use_artifact call. alignment_validated/
# _record_counts aren't known until after the fetch, so they're stamped via
# run.config.update(...) rather than the initial config= dict above.
stores, alignment_counts = fetch_and_validate_stores(
    CANDIDATE_MODEL_KEYS, lambda mk: PRODUCER_REFS[mk],
    run=run, dataset_sample=test_sample, render_task=SPEC.render_task,
    expected_dataset=DATASET, expected_data_seed=DATA_SEED,
    expected_eval_size=EVAL_SIZE, n_generators=5,
)
run.config.update({
    "alignment_validated": True,
    "alignment_record_counts": alignment_counts,
})
define_wandb_chart_only_metrics(prefixes=[f"{DATASET}_sample/*"])

bonmav_records = []
already_done = set()

if RESUME_FROM_REF:
    resumed_records, _ = load_generations_artifact(RESUME_FROM_REF, run=run)
    bonmav_records = resumed_records
    already_done = {r["problem_id"] for r in resumed_records}
    print(f"Resuming from {RESUME_FROM_REF}: {len(already_done)} problems already logged, skipping them.")
else:
    print("Starting fresh at problem_id=0.")

bonmav_tie_rng = random.Random(TIE_BREAK_SEED)
agent_ids = slot_agent_ids(ENSEMBLE)
n_aspects = len(ASPECT_IDS)
producer_ref = None


def _log_manifest():
    return {
        "verifier_key": VERIFIER_KEY, "dataset": DATASET,
        "candidate_ensemble": ENSEMBLE.ensemble_id, "aspect_subset": SUBSET_ID,
        "protocol": "BoN-MAV@3/M4", "data_seed": DATA_SEED, "eval_size": EVAL_SIZE,
        "wandb_run_id": run.id, "wandb_run_name": run.name, "wandb_project": PROJECT,
        "n_records": len(bonmav_records),
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }


for problem_id, item in enumerate(test_sample):
    if problem_id in already_done:
        continue

    slots = sample_slots(ENSEMBLE, dataset=DATASET, problem_id=problem_id, rep=0)
    candidate_specs = [
        AgentSpec(agent_id=aid, model_key=mk, model_id=get_model(mk)["model_id"], mode="replay",
                  store=stores[mk], gen_id=gid)
        for aid, (mk, gid) in zip(agent_ids, slots)
    ]
    graph = build_bon_mav_graph(candidate_specs, ASPECT_IDS, verify_fn, select_fn, rng=bonmav_tie_rng)
    task = SPEC.render_task(item)
    ground_truth = SPEC.get_truth(item)
    out = graph.invoke({
        "task": task, "problem_id": problem_id, "dataset": DATASET,
        "samples": [], "approvals": [], "selection": None,
    })

    selection = out["selection"]
    selected_idx = selection["selected_idx"]
    selected_sample = out["samples"][selected_idx]
    compare_value = selected_sample["answer"] if SPEC.compare_on == "answer" else selected_sample["parsed_output"]
    correct = SPEC.score(compare_value, ground_truth)

    # reused_gen_*/live_verify_* cost split computed separately on each
    # half, plus the combined total the results/matrices use.
    reused_metrics = compose_generation_metrics(out["samples"], tokens_per_s_rule="mean")
    live_metrics = compose_generation_metrics(out["approvals"], tokens_per_s_rule="mean")
    composed = compose_generation_metrics(out["samples"] + out["approvals"], tokens_per_s_rule="mean")

    record = {
        "dataset": DATASET, "problem_id": problem_id,
        "verifier_key": VERIFIER_KEY, "candidate_ensemble": ENSEMBLE.ensemble_id, "aspect_subset": SUBSET_ID,
        "protocol": "BoN-MAV@3/M4", "slots": [[mk, gid] for mk, gid in slots],
        "selected_idx": selected_idx, "scores": selection["scores"],
        "tie": selection["tie"], "tie_break_used": selection["tie_break_used"],
        "ground_truth": ground_truth, "final_output": selected_sample["parsed_output"], "correct": correct,
        "pct_approvals_top": selection["scores"][selected_idx] / n_aspects,
        "approvals": out["approvals"],
        "latency_s": composed["latency_s"], "output_tokens": composed["output_tokens"],
        "total_tokens": composed["total_tokens"], "gpu_peak_mb": composed["gpu_peak_mb"],
        "throughput_tok_s": composed["tokens_per_s"],
        "reused_gen_latency_s": reused_metrics["latency_s"], "reused_gen_total_tokens": reused_metrics["total_tokens"],
        "live_verify_latency_s": live_metrics["latency_s"], "live_verify_total_tokens": live_metrics["total_tokens"],
    }
    bonmav_records.append(record)

    log_metrics({
        f"{DATASET}_sample/accuracy": int(correct),
        f"{DATASET}_sample/latency_s": round(composed["latency_s"], 2),
        f"{DATASET}_sample/output_tokens": composed["output_tokens"],
        f"{DATASET}_sample/total_tokens": composed["total_tokens"],
        f"{DATASET}_sample/gpu_peak_mb": round(composed["gpu_peak_mb"], 2),
        f"{DATASET}_sample/throughput_tok_s": round(composed["tokens_per_s"], 4),
    })
    print(f"  problem={problem_id}: selected_idx={selected_idx} correct={correct} "
          f"tie={selection['tie']} scores={selection['scores']}")

    if len(bonmav_records) % LOG_EVERY_N_PROBLEMS == 0:
        artifact = log_generations_artifact(bonmav_records, _log_manifest(), name=VERDICTS_ARTIFACT_NAME, run=run)
        producer_ref = artifact_ref(artifact, entity=run.entity, project=PROJECT)
        wandb.summary["producer_artifact"] = producer_ref
        print(f"  ✅ re-logged verdicts artifact after {len(bonmav_records)} problems -> {producer_ref}")

# Final re-log to catch any tail not a multiple of LOG_EVERY_N_PROBLEMS.
artifact = log_generations_artifact(bonmav_records, _log_manifest(), name=VERDICTS_ARTIFACT_NAME, run=run)
producer_ref = artifact_ref(artifact, entity=run.entity, project=PROJECT)
wandb.summary["producer_artifact"] = producer_ref

print(f"\n✅ BoN-MAV generation complete: {len(bonmav_records)}/{EVAL_SIZE} problems logged -> {producer_ref}")
if len(bonmav_records) < EVAL_SIZE:
    print(
        f"⚠️  Only {len(bonmav_records)}/{EVAL_SIZE} problems completed this session. "
        f"Set RESUME_FROM_REF = {producer_ref!r} in §3 and re-run to finish the remaining "
        f"{EVAL_SIZE - len(bonmav_records)} problems."
    )

## 9. Final Statistics, Matrices Artifact, and Analysis

**Only meaningful once all `EVAL_SIZE` problems have been logged** (across however many sessions it took) — if this session stopped early, resume per the warning printed at the end of §8 and re-run this cell afterwards rather than treating a partial result as final.

In [ ]:
import gc
import tempfile
import numpy as np
import torch
from utils.statistics import (
    accuracy_statistics, time_statistics, token_statistics, vram_statistics,
    print_accuracy_stats_report, print_time_report, print_token_report, print_vram_report,
)
from utils.mav import approval_diagnostics

if len(bonmav_records) < EVAL_SIZE:
    print(f"⚠️  Only {len(bonmav_records)}/{EVAL_SIZE} problems completed — the analysis below "
          f"is PARTIAL, not the final result. Resume and re-run once all problems are done.")

sorted_records = sorted(bonmav_records, key=lambda r: r["problem_id"])
n_done = len(sorted_records)

correctness_matrix = np.zeros((n_done, 1), dtype=int)
time_matrix         = np.zeros((n_done, 1), dtype=float)
output_tok_matrix   = np.zeros((n_done, 1), dtype=int)
total_tok_matrix    = np.zeros((n_done, 1), dtype=int)
gpu_peak_matrix      = np.zeros((n_done, 1), dtype=float)

results_columns = [
    "dataset", "verifier", "candidate_ensemble", "aspect_subset", "protocol", "problem_id", "problem_ref",
    "slots", "selected_idx", "ground_truth", "final_output", "correct", "pct_approvals_top", "tie",
    "latency_s", "output_tokens", "total_tokens", "gpu_peak_mb", "throughput_tok_s",
]
results_rows = []
all_approvals = []

for i, r in enumerate(sorted_records):
    correctness_matrix[i, 0] = int(r["correct"])
    time_matrix[i, 0]        = r["latency_s"]
    output_tok_matrix[i, 0]  = r["output_tokens"]
    total_tok_matrix[i, 0]   = r["total_tokens"]
    gpu_peak_matrix[i, 0]    = r["gpu_peak_mb"]
    all_approvals.extend(r["approvals"])

    slots_str = ",".join(f"{mk}:{gid}" for mk, gid in r["slots"])
    results_rows.append([
        DATASET, VERIFIER_KEY, ENSEMBLE.ensemble_id, SUBSET_ID, r["protocol"], r["problem_id"],
        problem_ref(DATASET, r["problem_id"]), slots_str, r["selected_idx"], r["ground_truth"],
        r["final_output"], r["correct"], round(r["pct_approvals_top"], 4), r["tie"],
        round(r["latency_s"], 2), r["output_tokens"], r["total_tokens"],
        round(r["gpu_peak_mb"], 2), round(r["throughput_tok_s"], 4),
    ])

results_table = wandb.Table(columns=results_columns, data=results_rows)
wandb.log({f"results_bonmav_{VERIFIER_KEY}_{ENSEMBLE.ensemble_id}_{SUBSET_ID}": results_table})

acc_stats   = accuracy_statistics(correctness_matrix)
time_stats  = time_statistics(time_matrix)
out_stats   = token_statistics(output_tok_matrix)
total_stats = token_statistics(total_tok_matrix)
vram_stats  = vram_statistics(gpu_peak_matrix)

print_accuracy_stats_report(f"BoN-MAV {VERIFIER_KEY}/{ENSEMBLE.ensemble_id}/{SUBSET_ID} Accuracy ({DATASET})", acc_stats)
print_time_report(f"BoN-MAV {VERIFIER_KEY}/{ENSEMBLE.ensemble_id}/{SUBSET_ID} Latency ({DATASET})", time_stats)
print_vram_report(f"BoN-MAV {VERIFIER_KEY}/{ENSEMBLE.ensemble_id}/{SUBSET_ID} Peak VRAM ({DATASET})", vram_stats)

n_tie = sum(1 for r in sorted_records if r["tie"])
diag = approval_diagnostics(all_approvals, n_candidates=len(ENSEMBLE.slots))
overall_parse_rate = sum(1 for a in all_approvals if a["verdict"] is not None) / len(all_approvals)

wandb.summary["accuracy"]            = round(acc_stats["p_hat"], 6)
wandb.summary["selection_tie_rate"]  = round(n_tie / n_done, 4)
wandb.summary["overall_parse_rate"]  = round(overall_parse_rate, 4)
wandb.summary["mean_vram_peak"]      = round(vram_stats["mean_peak"], 0)
wandb.summary["max_vram_peak"]       = round(vram_stats["peak_max"], 0)
wandb.summary["n_problems_completed"] = n_done

for aspect_id, stats in diag.items():
    wandb.summary[f"approval_rate/{aspect_id}"] = round(stats["approval_rate"], 4)
    wandb.summary[f"parse_rate/{aspect_id}"] = round(stats["parse_rate"], 4)
    print(f"  {aspect_id}: approval_rate={stats['approval_rate']:.1%} parse_rate={stats['parse_rate']:.1%}")
    if stats["approval_rate"] > 0.95:
        print(f"    ⚠️  sycophancy flag: {aspect_id} approves >95% of candidates")

mean_reused_latency = sum(r["reused_gen_latency_s"] for r in sorted_records) / n_done
mean_live_latency = sum(r["live_verify_latency_s"] for r in sorted_records) / n_done
wandb.summary["mean_reused_gen_latency_s"]   = round(mean_reused_latency, 2)
wandb.summary["mean_live_verify_latency_s"]  = round(mean_live_latency, 2)

log_stat_summary("accuracy_stats", acc_stats, keys=["ci_lower", "ci_upper", "between_problem_std"])
log_stat_summary("latency_stats", time_stats, keys=["ci_lower", "ci_upper"])
log_stat_summary("output_tokens_stats", out_stats, keys=["ci_lower", "ci_upper"])
log_stat_summary("total_tokens_stats", total_stats, keys=["ci_lower", "ci_upper"])
log_stat_summary("vram_peak_stats", vram_stats, keys=["peak_ci_lower", "peak_ci_upper"])

# ── Evaluation-matrices artifact ────────────────────────────────────────────
artifact = wandb.Artifact(
    bonmav_matrices_artifact_name(VERIFIER_KEY, ENSEMBLE.ensemble_id, SUBSET_ID, DATASET),
    type="evaluation-data",
    description=f"BoN-MAV {VERIFIER_KEY}/{ENSEMBLE.ensemble_id}/{SUBSET_ID} evaluation matrices "
                f"({n_done} x 1). {DATASET}.",
)
with tempfile.NamedTemporaryFile(suffix=".npz", delete=False) as f:
    np.savez(
        f.name,
        problem_ids=np.array([r["problem_id"] for r in sorted_records]),
        bonmav_correctness=correctness_matrix,
        bonmav_times=time_matrix,
        bonmav_output_tokens=output_tok_matrix,
        bonmav_total_tokens=total_tok_matrix,
        bonmav_gpu_peak=gpu_peak_matrix,
    )
    artifact.add_file(f.name, name="evaluation_matrices.npz")
wandb.log_artifact(artifact)

# ── Unload verifier ──────────────────────────────────────────────────────────
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
final_allocated_mb = torch.cuda.memory_allocated() / 1e6
print(f"\nVRAM after unloading verifier: {final_allocated_mb:.1f} MB")

finish_wandb()
print(f"\n✅ BoN-MAV {VERIFIER_KEY}/{ENSEMBLE.ensemble_id}/{SUBSET_ID}/{DATASET} run finished: "
      f"accuracy={acc_stats['p_hat']:.3f} ({n_done}/{EVAL_SIZE} problems)")